In [ ]:
"""
Author: Aaron Niecestro
Project: Course Recommender System Capstone

Created: November 4, 2026
Last Edit: November 4, 2026
Progress: Complete

Analysis: Non-Negative Matrix Factorization (NMF)

Method:
Latent Factor Collaborative Filtering
Using Matrix Decomposition

Description:

Builds a recommendation system using
Non-Negative Matrix Factorization (NMF).

The user-item interaction matrix is decomposed
into lower-dimensional latent user and item
feature matrices. Predicted ratings are generated
through matrix reconstruction and evaluated using
Root Mean Squared Error (RMSE).

Multiple latent factor dimensions are evaluated
to determine the optimal matrix factorization
configuration.

Techniques Applied:

- Collaborative Filtering
- Matrix Factorization
- Dimensionality Reduction
- Non-Negative Matrix Factorization
- Latent Feature Learning
- Hyperparameter Optimization
- Train-Test Split Validation
- RMSE Evaluation
- Recommendation System Analysis

Datasets:

- ratings.csv

Research Questions:

- Can matrix factorization accurately predict
  user-course ratings?

- What number of latent factors produces the
  best recommendation performance?

- Can latent feature learning reduce
  dimensionality while maintaining accuracy?

- How does recommendation accuracy change as
  the number of latent factors increases?

- What latent dimension minimizes RMSE?

Outputs:

- nmf_results.csv
- model_comparison.csv
- nmf_rmse_comparison.png
- latent_factor_comparison.png

Evaluation Metrics:

- RMSE
- Latent Factor Comparison
- Model Ranking
"""

============================================================
Imports
============================================================

In [ ]:
import random
import warnings

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
from pathlib import Path

In [ ]:
from sklearn.decomposition import NMF

In [ ]:
from sklearn.model_selection import (
    train_test_split
)

In [ ]:
from sklearn.metrics import (
    mean_squared_error
)

In [ ]:
warnings.filterwarnings("ignore")

============================================================
Reproducibility
============================================================

In [ ]:
SEED = 423

In [ ]:
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
RANDOM_STATE = SEED

============================================================
Paths
============================================================

In [ ]:
DATA_DIR = Path(
    r"C:\Users\aniec\Desktop\ibm-ml-project\capstone\00-Data"
)

In [ ]:
OUTPUT_DIR = Path(
    r"C:\Users\aniec\Desktop\ibm-ml-project\capstone\03-Supervised\supervised-results"
)

In [ ]:
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

============================================================
Load Data
============================================================

In [ ]:
rating_df = pd.read_csv(
    DATA_DIR / "ratings.csv"
)

In [ ]:
print("\nDataset Loaded Successfully")
print("-" * 40)

In [ ]:
print(
    f"Dataset Shape : "
    f"{rating_df.shape}"
)

============================================================
Dataset Summary
============================================================

In [ ]:
print("\nDataset Summary")
print("-" * 40)

In [ ]:
print(
    f"Total Ratings : "
    f"{len(rating_df):,}"
)

In [ ]:
print(
    f"Unique Users : "
    f"{rating_df['user'].nunique():,}"
)

In [ ]:
print(
    f"Unique Courses : "
    f"{rating_df['item'].nunique():,}"
)

In [ ]:
print("\nRating Distribution")
print("-" * 40)

In [ ]:
print(
    rating_df["rating"]
    .value_counts()
    .sort_index()
)

============================================================
Train-Test Split
============================================================

In [ ]:
train_df, test_df = train_test_split(
    rating_df,
    test_size=0.30,
    random_state=RANDOM_STATE
)

In [ ]:
print("\nTrain/Test Split")
print("-" * 40)

In [ ]:
print(
    f"Training Ratings : "
    f"{len(train_df):,}"
)

In [ ]:
print(
    f"Testing Ratings  : "
    f"{len(test_df):,}"
)

============================================================
Training User-Item Matrix
============================================================

In [ ]:
train_matrix = (
    train_df.pivot_table(
        index="user",
        columns="item",
        values="rating",
        fill_value=0
    )
)

In [ ]:
print("\nTraining Matrix")
print("-" * 40)

In [ ]:
print(
    f"Matrix Shape : "
    f"{train_matrix.shape}"
)

============================================================
NMF Hyperparameter Search
============================================================

In [ ]:
latent_factors = [
    4,
    8,
    16,
    24,
    32,
    48,
    64
]

In [ ]:
results = []

In [ ]:
global_mean = (
    train_df["rating"]
    .mean()
)

In [ ]:
print("\nGlobal Mean Rating")
print("-" * 40)

In [ ]:
print(
    f"{global_mean:.4f}"
)

============================================================
Baseline RMSE
============================================================

In [ ]:
sample_test = test_df.sample(
    n=min(5000, len(test_df)),
    random_state=RANDOM_STATE
)

In [ ]:
baseline_predictions = [
    global_mean
] * len(sample_test)

In [ ]:
baseline_rmse = np.sqrt(
    mean_squared_error(
        sample_test["rating"],
        baseline_predictions
    )
)

In [ ]:
print("\nBaseline RMSE")
print("-" * 40)

In [ ]:
print(
    f"{baseline_rmse:.4f}"
)

============================================================
NMF Evaluation
============================================================

In [ ]:
print("\nNMF Evaluation")
print("-" * 40)

In [ ]:
for n_components in latent_factors:

    nmf_model = NMF(
        n_components=n_components,
        init="random",
        random_state=RANDOM_STATE,
        max_iter=500
    )

    W = nmf_model.fit_transform(
        train_matrix
    )

    H = nmf_model.components_

    reconstructed_matrix = np.dot(
        W,
        H
    )

    reconstructed_df = pd.DataFrame(
        reconstructed_matrix,
        index=train_matrix.index,
        columns=train_matrix.columns
    )

    predictions = []
    actuals = []

    sample_test = test_df.sample(
        n=min(
            5000,
            len(test_df)
        ),
        random_state=RANDOM_STATE
    )

    for row in sample_test.itertuples():

        if (
            row.user in reconstructed_df.index
            and
            row.item in reconstructed_df.columns
        ):

            prediction = (
                reconstructed_df.loc[
                    row.user,
                    row.item
                ]
            )

        else:

            prediction = global_mean

        predictions.append(
            prediction
        )

        actuals.append(
            row.rating
        )

    rmse = np.sqrt(
        mean_squared_error(
            actuals,
            predictions
        )
    )

    results.append([
        n_components,
        rmse
    ])

    print(
        f"Latent Factors = "
        f"{n_components:<2} | "
        f"RMSE = {rmse:.4f}"
    )
    
    print(
        f"Min Prediction : "
        f"{reconstructed_df.min().min():.4f}"
    )

    print(
        f"Max Prediction : "
        f"{reconstructed_df.max().max():.4f}"
    )

============================================================
Results DataFrame
============================================================

In [ ]:
results_df = pd.DataFrame(
    results,
    columns=[
        "Latent_Factors",
        "RMSE"
    ]
)

In [ ]:
results_df = (
    results_df
    .sort_values(
        by="RMSE",
        ascending=True
    )
)

In [ ]:
print("\nNMF Results")
print("-" * 40)

In [ ]:
print(
    results_df
)

============================================================
Best Configuration
============================================================

In [ ]:
best_result = (
    results_df.iloc[0]
)

In [ ]:
best_factors = int(
    best_result["Latent_Factors"]
)

In [ ]:
best_rmse = (
    best_result["RMSE"]
)

In [ ]:
print("\nBest Configuration")
print("-" * 40)

In [ ]:
print(
    f"Latent Factors : "
    f"{best_factors}"
)

In [ ]:
print(
    f"RMSE           : "
    f"{best_rmse:.4f}"
)

============================================================
Matrix Sparsity Analysis
============================================================

In [ ]:
total_cells = (
    train_matrix.shape[0]
    * train_matrix.shape[1]
)

In [ ]:
non_zero_cells = (
    train_matrix != 0
).sum().sum()

In [ ]:
sparsity = (
    1 -
    (
        non_zero_cells /
        total_cells
    )
) * 100

In [ ]:
print("\nMatrix Sparsity")
print("-" * 40)

In [ ]:
print(
    f"Total Cells      : "
    f"{total_cells:,}"
)

In [ ]:
print(
    f"Observed Ratings : "
    f"{non_zero_cells:,}"
)

In [ ]:
print(
    f"Sparsity (%)     : "
    f"{sparsity:.2f}"
)

============================================================
Results DataFrame
============================================================

In [ ]:
results_df = pd.DataFrame(
    results,
    columns=[
        "Latent_Factors",
        "RMSE"
    ]
)

In [ ]:
results_df = (
    results_df
    .sort_values(
        by="RMSE",
        ascending=True
    )
)

In [ ]:
print("\nNMF Results")
print("-" * 40)

In [ ]:
print(results_df)

============================================================
Baseline Comparison
============================================================

In [ ]:
rmse_difference = (
    best_rmse -
    baseline_rmse
)

In [ ]:
performance_gap = (
    (best_rmse - baseline_rmse)
    /
    baseline_rmse
) * 100

In [ ]:
print("\nBaseline Comparison")
print("-" * 40)

In [ ]:
print(
    f"Baseline RMSE : "
    f"{baseline_rmse:.4f}"
)

In [ ]:
print(
    f"NMF RMSE      : "
    f"{best_rmse:.4f}"
)

In [ ]:
print(
    f"Difference    : "
    f"{rmse_difference:.4f}"
)

In [ ]:
print(
    f"Gap (%)       : "
    f"{performance_gap:.2f}%"
)

============================================================
Visualization 1
RMSE vs Latent Factors
============================================================

In [ ]:
plt.figure(figsize=(10, 6))

In [ ]:
sns.lineplot(
    data=results_df,
    x="Latent_Factors",
    y="RMSE",
    marker="o",
    color="#1f77b4"
)

In [ ]:
plt.title(
    "NMF RMSE by Number of Latent Factors"
)

In [ ]:
plt.xlabel(
    "Latent Factors"
)

In [ ]:
plt.ylabel(
    "RMSE"
)

In [ ]:
plt.grid(True)

In [ ]:
plt.tight_layout()

In [ ]:
plt.savefig(
    OUTPUT_DIR /
    "nmf_rmse_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

In [ ]:
plt.show()

============================================================
Visualization 2
NMF Configuration Comparison
============================================================

In [ ]:
plt.figure(figsize=(10, 6))

In [ ]:
sns.barplot(
    data=results_df,
    x="Latent_Factors",
    y="RMSE",
    color="#1f77b4"
)

In [ ]:
plt.title(
    "NMF Performance by Latent Dimension"
)

In [ ]:
plt.xlabel(
    "Latent Factors"
)

In [ ]:
plt.ylabel(
    "RMSE"
)

In [ ]:
plt.tight_layout()

In [ ]:
plt.savefig(
    OUTPUT_DIR /
    "latent_factor_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

In [ ]:
plt.show()

============================================================
Visualization 3
NMF vs Baseline
============================================================

In [ ]:
comparison_df = pd.DataFrame({
    "Model": [
        "Baseline",
        "Best NMF"
    ],
    "RMSE": [
        baseline_rmse,
        best_rmse
    ]
})

In [ ]:
plt.figure(figsize=(8, 5))

In [ ]:
sns.barplot(
    data=comparison_df,
    x="Model",
    y="RMSE",
    palette=[
        "#ff7f0e",
        "#1f77b4"
    ]
)

In [ ]:
plt.title(
    "Baseline vs NMF"
)

In [ ]:
plt.xlabel(
    "Model"
)

In [ ]:
plt.ylabel(
    "RMSE"
)

In [ ]:
plt.tight_layout()

In [ ]:
plt.savefig(
    OUTPUT_DIR /
    "nmf_baseline_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

In [ ]:
plt.show()

============================================================
Final Summary Table
============================================================

In [ ]:
summary_df = pd.DataFrame({
    "Metric": [
        "Best Latent Factors",
        "Best RMSE",
        "Baseline RMSE",
        "Matrix Sparsity (%)",
        "Users",
        "Courses",
        "Ratings"
    ],
    "Value": [
        int(best_factors),
        round(best_rmse, 4),
        round(baseline_rmse, 4),
        round(sparsity, 2),
        f"{rating_df['user'].nunique():,}",
        f"{rating_df['item'].nunique():,}",
        f"{len(rating_df):,}"
    ]
})

In [ ]:
print("\nFinal Summary")
print("-" * 40)

In [ ]:
print(
    summary_df.to_string(
        index=False
    )
)

In [ ]:
print("\nFinding")
print("-" * 40)

In [ ]:
print(
    "Standard NMF struggled with the "
    "95.8% sparse user-item matrix and "
    "performed substantially worse than "
    "the baseline predictor."
)

In [ ]:
results_df.to_csv(
    OUTPUT_DIR /
    "nmf_results.csv",
    index=False
)

In [ ]:
"""
Best Configuration
• Latent Factors: 4
• RMSE: 3.2763

Key Findings
• User-item matrix sparsity reached 95.8%.
• NMF performance deteriorated as latent factors increased.
• Standard NMF was unable to effectively model the sparse interaction matrix.
• Baseline RMSE (0.8187) substantially outperformed NMF RMSE (3.2763).
• Item-Based KNN with MSD similarity remained the strongest recommendation approach.
"""

In [ ]:
"""
Findings

The user-item matrix exhibited 95.8% sparsity.
Standard Non-Negative Matrix Factorization struggled to reconstruct user preferences effectively.
Increasing latent factors increased prediction error.
The best NMF configuration used 4 latent factors and achieved an RMSE of 3.2763.
NMF underperformed the baseline predictor (RMSE = 0.8187).
Matrix factorization was less effective than item-based KNN collaborative filtering for this dataset.

Conclusion

NMF was sensitive to the extreme sparsity of the interaction matrix.
Additional latent factors did not improve recommendation quality.
Item-Based KNN with MSD similarity remained the strongest recommendation approach evaluated in the project.
"""